# Netherlands Lottery Prediction System

Interactive notebook for training, prediction, and backtest analysis.

**Master Strategy**: Anchor-cluster framework with constraint-aware ticket generation.

- Load historical draw data
- Train transformer model
- Generate tickets (coverage + convergence)
- Evaluate performance via backtest

## 1. Setup & Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_pipeline import LottoData, build_sequence_dataset
from ml_model import build_model, train_model, predict_probs
from constraint_generator import TicketGenerator, TicketConfig
from backtest_engine import BacktestEngine

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All modules imported successfully")

## 2. Load & Explore Data

In [ ]:
# Load data
CSV_PATH = "nl_lotto_xl_history.csv"
GAME = "xl"

data = LottoData(CSV_PATH, game=GAME)
df = data.get_df()

print(f"Loaded {len(df)} draws")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nLast draw:")
print(data.get_last_draw())
print(f"\nFirst 5 rows:")
print(df.head())

## 3. Analyze Ball Statistics

In [ ]:
# Compute stats
freq = data.compute_frequency_stats()
gaps = data.compute_gap_stats()
hot, cold = data.compute_hot_cold(recent_window=20)

print(f"Hot numbers (recent, high freq): {sorted(hot)}")
print(f"Cold numbers (overdue): {sorted(cold)}")

# Frequency distribution
freq_vals = list(freq.values())
print(f"\nFrequency Stats:")
print(f"  Mean: {np.mean(freq_vals):.4f}")
print(f"  Std: {np.std(freq_vals):.4f}")
print(f"  Min: {np.min(freq_vals):.4f}")
print(f"  Max: {np.max(freq_vals):.4f}")

# Top/bottom balls
top_freq = sorted(freq.items(), key=lambda x: -x[1])[:10]
bottom_freq = sorted(freq.items(), key=lambda x: x[1])[:10]

print(f"\nTop 10 by frequency:")
for ball, f in top_freq:
    print(f"  {ball}: {f:.4f}")

print(f"\nBottom 10 by frequency:")
for ball, f in bottom_freq:
    print(f"  {ball}: {f:.4f}")

### Visualize Frequency Distribution

In [ ]:
# Plot frequency by ball
balls = list(range(1, 46))
freqs = [freq[b] for b in balls]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Histogram
axes[0].bar(balls, freqs, color=['red' if b in hot else 'blue' if b in cold else 'gray' for b in balls], alpha=0.7)
axes[0].axhline(np.mean(freqs), color='k', linestyle='--', label='Mean')
axes[0].set_xlabel('Ball Number')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Ball Frequency Distribution (Red=Hot, Blue=Cold)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gap distribution
gap_vals = [gaps[b] for b in balls]
axes[1].bar(balls, gap_vals, color=['blue' if b in cold else 'gray' for b in balls], alpha=0.7)
axes[1].set_xlabel('Ball Number')
axes[1].set_ylabel('Gaps Since Last Draw')
axes[1].set_title('Ball Gaps (Blue=Overdue)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Hot/Cold classification complete")

## 4. Build Sequence Dataset

In [ ]:
# Build sequences
LOOKBACK = 20
X, y_main, y_res, dates = build_sequence_dataset(df, lookback=LOOKBACK)

print(f"Dataset shapes:")
print(f"  X: {X.shape} (sequences, lookback, balls)")
print(f"  y_main: {y_main.shape} (sequences, balls)")
print(f"  y_res: {y_res.shape} (sequences, balls)")

print(f"\nSequence count: {len(X)}")
print(f"Effective lookback: {X.shape[1]}")

## 5. Train Model

In [ ]:
# Train/val split
VAL_SIZE = 24
val_size_eff = min(VAL_SIZE, max(1, len(X) // 4))
i_split = max(1, len(X) - val_size_eff)

X_train, X_val = X[:i_split], X[i_split:]
y_main_train, y_main_val = y_main[:i_split], y_main[i_split:]
y_res_train, y_res_val = y_res[:i_split], y_res[i_split:]

print(f"Train set: {len(X_train)} sequences")
print(f"Validation set: {len(X_val)} sequences")

# Build model
model = build_model(lookback=X.shape[1])
print(f"\n✓ Model built")

In [ ]:
# Train
print("Training...")
history = train_model(
    model,
    X_train, y_main_train, y_res_train,
    X_val, y_main_val, y_res_val,
    epochs=20,
    batch_size=32,
    verbose=1
)

print(f"\n✓ Training complete")

### Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reserve Accuracy
if 'reserve_accuracy' in history.history:
    axes[1].plot(history.history['reserve_accuracy'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_reserve_accuracy'], label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Reserve Prediction Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Predict Probabilities & Generate Tickets

In [ ]:
# Predict on last sequence
main_probs, reserve_probs = predict_probs(model, X[-1:], verbose=0)
main_probs = main_probs[0]  # (45,)
reserve_probs = reserve_probs[0]  # (45,)

print(f"Main probabilities: {main_probs.shape}")
print(f"Reserve probabilities: {reserve_probs.shape}")

# Top balls
top_idx = np.argsort(-main_probs)[:15] + 1
print(f"\nTop 15 balls by main probability:")
for ball in top_idx:
    print(f"  {ball}: {main_probs[ball-1]:.4f}")

In [ ]:
# Generate tickets
gen = TicketGenerator(config=TicketConfig(game=GAME))

tickets_dict = gen.generate(
    main_probs,
    num_coverage=16,
    num_convergence=8,
    hot_numbers=hot,
    cold_numbers=cold
)

coverage_tickets = tickets_dict['coverage']
convergence_tickets = tickets_dict['convergence']

print(f"Generated {len(coverage_tickets)} coverage + {len(convergence_tickets)} convergence tickets")
print(f"\nCOVERAGE TICKETS (broad statistical coverage):")
for i, ticket in enumerate(coverage_tickets, 1):
    prob_avg = np.mean([main_probs[n-1] for n in ticket])
    print(f"  {i:02d}) {ticket} (avg prob: {prob_avg:.4f})")

print(f"\nCONVERGENCE TICKETS (aggressive, high-variance):")
for i, ticket in enumerate(convergence_tickets, 1):
    prob_avg = np.mean([main_probs[n-1] for n in ticket])
    print(f"  {i:02d}) {ticket} (avg prob: {prob_avg:.4f})")

## 7. Quick Backtest (Last 30 Draws)

In [ ]:
# Simple backtest: evaluate generated tickets against test set
print("Quick validation against last few draws...\n")

all_tickets = coverage_tickets + convergence_tickets

# Check against last 5 actual draws
test_draws = df.iloc[-5:].reset_index(drop=True)

for idx, (_, draw_row) in enumerate(test_draws.iterrows()):
    actual_main = data.extract_main_numbers(draw_row)
    actual_reserve = data.extract_reserve(draw_row)
    date = draw_row.get('date', idx)
    
    matches = []
    for ticket in all_tickets:
        ticket_set = set(ticket)
        match_count = len(ticket_set & actual_main)
        matches.append(match_count)
    
    best_match = max(matches) if matches else 0
    avg_match = np.mean(matches) if matches else 0
    
    print(f"[{date}] Main: {sorted(actual_main)}, Reserve: {actual_reserve}")
    print(f"  Best ticket match: {best_match}, Avg: {avg_match:.2f}")
    print(f"  Tickets with 3+ matches: {sum(1 for m in matches if m >= 3)}")
    print()

## 8. Full Rolling Backtest (Optional - Takes Time)

In [ ]:
# Run rolling backtest (this takes a while)
print("Running rolling backtest...")
print("(This may take 5-10 minutes)\n")

engine = BacktestEngine(game=GAME)

# Only test last 30 draws for speed
test_start = max(LOOKBACK + 5, len(df) - 30)
print(f"Testing draws {test_start} to {len(df)}\n")

for t in range(test_start, len(df)):
    df_train = df.iloc[:t]
    df_eval = df.iloc[t]
    
    try:
        # Build sequences
        X_t, y_main_t, y_res_t, _ = build_sequence_dataset(df_train, lookback=LOOKBACK)
        
        if len(X_t) < 2:
            continue
        
        # Quick train
        val_eff = min(VAL_SIZE, max(1, len(X_t) // 4))
        i_sp = max(1, len(X_t) - val_eff)
        
        model_t = build_model(lookback=X_t.shape[1])
        train_model(
            model_t,
            X_t[:i_sp], y_main_t[:i_sp], y_res_t[:i_sp],
            X_t[i_sp:], y_main_t[i_sp:], y_res_t[i_sp:],
            epochs=10, batch_size=32, verbose=0
        )
        
        # Predict & generate
        m_probs, _ = predict_probs(model_t, X_t[-1:], verbose=0)
        m_probs = m_probs[0]
        
        tickets_t = gen.generate(m_probs, num_coverage=16, num_convergence=8, 
                                hot_numbers=hot, cold_numbers=cold)
        all_t = tickets_t['coverage'] + tickets_t['convergence']
        
        # Evaluate
        actual = {
            'date': str(df_eval.get('date', t)),
            'main': data.extract_main_numbers(df_eval),
            'reserve': data.extract_reserve(df_eval)
        }
        
        result = engine.evaluate_set(all_t, actual)
        print(f"[{t}/{len(df)}] {result.date}: best={result.best_match}, 3+={result.hits_3}")
        
    except Exception as e:
        print(f"[{t}/{len(df)}] ERROR: {e}")
        continue

print(f"\n✓ Backtest complete")

In [ ]:
# Summary
metrics = engine.compute_metrics()
engine.print_summary()

print(f"Key metrics:")
print(f"  3-hit coverage: {metrics.coverage_3*100:.1f}%")
print(f"  4-hit coverage: {metrics.coverage_4*100:.1f}%")
print(f"  5-hit coverage: {metrics.coverage_5*100:.1f}%")
print(f"  Average matches per ticket: {metrics.avg_match:.2f}")

## 9. Results Visualization

In [ ]:
# Get results as DataFrame
results_df = engine.get_results_df()

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Best match distribution
axes[0, 0].hist(results_df['best_match'], bins=7, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Best Match per Draw')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Best Ticket Matches')
axes[0, 0].grid(True, alpha=0.3)

# Hits over time
axes[0, 1].plot(results_df['best_match'], marker='o', label='Best match')
axes[0, 1].axhline(np.mean(results_df['best_match']), color='r', linestyle='--', label='Mean')
axes[0, 1].set_xlabel('Draw (chronological)')
axes[0, 1].set_ylabel('Match Count')
axes[0, 1].set_title('Best Matches Over Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Hit tier counts
hit_tiers = {
    '3-hit': results_df['hits_3'].sum(),
    '4-hit': results_df['hits_4'].sum(),
    '5-hit': results_df['hits_5'].sum(),
    '6-hit': results_df['hits_6'].sum()
}
axes[1, 0].bar(hit_tiers.keys(), hit_tiers.values(), edgecolor='black', alpha=0.7)
axes[1, 0].set_ylabel('Total Count')
axes[1, 0].set_title('Hit Tier Summary')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Coverage percentages
coverage_rates = {
    '≥3-hit': metrics.coverage_3*100,
    '≥4-hit': metrics.coverage_4*100,
    '≥5-hit': metrics.coverage_5*100,
    '6-hit': metrics.coverage_6*100
}
axes[1, 1].bar(coverage_rates.keys(), coverage_rates.values(), edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_ylabel('% of Draws with at Least 1 Hit')
axes[1, 1].set_title('Coverage Rates')
axes[1, 1].set_ylim([0, 100])
axes[1, 1].grid(True, alpha=0.3, axis='y')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 10. Summary & Next Steps

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║           LOTTERY PREDICTION SYSTEM - SUMMARY                  ║
╚════════════════════════════════════════════════════════════════╝

✓ Model Architecture:
  - Transformer-based encoder with Conv1D + attention
  - Multi-task learning: main (sigmoid) + reserve (softmax)
  - Weighted BCE for class imbalance, auxiliary sum-6 regularizer

✓ Strategy:
  - Anchor-cluster framework (XL: 4 pairs, Lotto: 5 numbers)
  - Constraint enforcement: 3 odd/3 even, low/mid/high spread
  - Two-tier: coverage (70%) + convergence (30%)
  - Hot/cold dynamic adjustment

✓ Key Metrics:
  - Average match per ticket: {:.2f}
  - 3-hit coverage: {:.1f}%
  - 4-hit coverage: {:.1f}%
  - 5-hit coverage: {:.1f}%

⚡ Next Steps:
  1. Export tickets for purchase
  2. Monitor actual draw results
  3. Update CSV weekly with new draws
  4. Re-run backtest to measure improvement
  5. Tune hyperparameters (epochs, lookback, anchor bias)

📊 Files Generated:
  - main.py: Full pipeline CLI
  - README.md: Complete documentation
  - *_history.csv: Historical data

💡 Pro Tips:
  - Increase num_convergence for higher-variance targeting
  - Use --start_tail 50 in backtest for speed
  - Try lookback=25-30 for longer memory
  - Run backtest weekly to track improvements
""".format(
    metrics.avg_match,
    metrics.coverage_3*100,
    metrics.coverage_4*100,
    metrics.coverage_5*100
))